# Automatron core

Configuration, schemas, provider routing, retrieval, the agent graph, and the
interfaces that sit on top of them. Sector packs import everything they need
from this module.

## Contents

1. Settings and configuration
2. Schemas
3. Sector registry
4. Provider router
5. Retrieval layer
6. Tool-agent loop
7. Graph builder
8. Verifier, rendering and audit
9. Run service
10. HTTP API
11. User interface

## 1. Settings and configuration

Pydantic settings, YAML config loading, logging with secret and PII redaction.

In [ ]:
import functools
import pathlib
from typing import Any, Literal

import yaml
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

SECTOR_ORDER = ("space", "quant", "ecommerce", "realestate")
PROVIDER_ORDER = ("gemini", "groq", "cerebras", "openrouter")
AGENT_ROLES = ("coordinator", "researcher", "analyst", "executor")


def find_root() -> pathlib.Path:
    """Locate the repository root from either a notebook or the generated module.

    The built module sits one level below the root and the notebooks two, so walk
    upwards until the config directory appears rather than hardcoding a depth.
    """
    if "__file__" in globals():
        start = pathlib.Path(globals()["__file__"]).resolve().parent
    else:
        start = pathlib.Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "config" / "sectors.yaml").is_file():
            return candidate
    return start


ROOT = find_root()


class Settings(BaseSettings):
    """Runtime configuration read from the environment and an optional .env file."""

    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
        case_sensitive=False,
    )

    groq_api_key: SecretStr | None = None
    gemini_api_key: SecretStr | None = None
    openrouter_api_key: SecretStr | None = None
    cerebras_api_key: SecretStr | None = None

    groq_model: str = ""
    gemini_model: str = ""
    cerebras_model: str = ""
    openrouter_models: str = ""
    openrouter_app_name: str = "Automatron"
    openrouter_site_url: str = ""

    qdrant_url: str = ""
    qdrant_api_key: SecretStr | None = None
    embed_model: str = "BAAI/bge-small-en-v1.5"

    port: int = 7860
    log_level: str = "INFO"
    app_username: str = ""
    app_password: SecretStr | None = None
    automatron_fake_llm: bool = False
    data_dir: pathlib.Path = pathlib.Path("./data")
    runtime_dir: pathlib.Path = pathlib.Path("/tmp/automatron")
    max_upload_mb: int = 10
    max_parallel_steps: int = 2
    rate_limit_per_ip_per_hour: int = 30

    trade_approval_threshold_usd: float | None = None
    ecom_high_value_threshold_usd: float | None = None

    def provider_keys(self) -> dict[str, SecretStr | None]:
        return {
            "gemini": self.gemini_api_key,
            "groq": self.groq_api_key,
            "cerebras": self.cerebras_api_key,
            "openrouter": self.openrouter_api_key,
        }

    def has_key(self, provider: str) -> bool:
        secret = self.provider_keys().get(provider)
        return bool(secret and secret.get_secret_value().strip())

    @property
    def configured_providers(self) -> list[str]:
        return [name for name in PROVIDER_ORDER if self.has_key(name)]

    @property
    def fake_mode(self) -> bool:
        """Fake mode is either requested outright or implied by having no keys at all."""
        return self.automatron_fake_llm or not self.configured_providers

    @property
    def model_overrides(self) -> dict[str, Any]:
        """Model ids supplied by the environment, which win over the YAML defaults."""
        overrides: dict[str, Any] = {}
        for provider, value in (
            ("gemini", self.gemini_model),
            ("groq", self.groq_model),
            ("cerebras", self.cerebras_model),
        ):
            if value.strip():
                overrides[provider] = value.strip()
        listed = [m.strip() for m in self.openrouter_models.split(",") if m.strip()]
        if listed:
            overrides["openrouter"] = listed
        return overrides

    def _absolute(self, value: pathlib.Path) -> pathlib.Path:
        return value if value.is_absolute() else (ROOT / value).resolve()

    @property
    def data_path(self) -> pathlib.Path:
        return self._absolute(self.data_dir)

    @property
    def runtime_path(self) -> pathlib.Path:
        return self._absolute(self.runtime_dir)

    @property
    def max_upload_bytes(self) -> int:
        return self.max_upload_mb * 1024 * 1024


@functools.lru_cache(maxsize=1)
def get_settings() -> Settings:
    """Return the process-wide settings, read once."""
    return Settings()


def reset_settings_cache() -> None:
    """Drop the cached settings so a test can change the environment and reload."""
    get_settings.cache_clear()
    load_provider_config.cache_clear()
    load_sector_config.cache_clear()

In [ ]:
CONFIG_DIR = ROOT / "config"


def _read_yaml(path: pathlib.Path) -> dict[str, Any]:
    if not path.is_file():
        raise FileNotFoundError(f"missing configuration file: {path}")
    with path.open(encoding="utf-8") as handle:
        loaded = yaml.safe_load(handle)
    if not isinstance(loaded, dict):
        raise ValueError(f"{path.name} must contain a mapping at the top level")
    return loaded


@functools.lru_cache(maxsize=1)
def load_provider_config() -> dict[str, Any]:
    """Provider definitions, role chains, and quota reservations."""
    config = _read_yaml(CONFIG_DIR / "providers.yaml")
    providers = config.get("providers") or {}
    roles = config.get("roles") or {}

    missing_roles = [role for role in AGENT_ROLES if role not in roles]
    if missing_roles:
        raise ValueError(f"providers.yaml is missing role chains for: {missing_roles}")
    for role, chain in roles.items():
        unknown = [name for name in chain if name not in providers]
        if unknown:
            raise ValueError(f"role '{role}' names providers that do not exist: {unknown}")

    for provider, override in get_settings().model_overrides.items():
        if provider not in providers:
            continue
        if isinstance(override, list):
            providers[provider]["models"] = override
        else:
            providers[provider]["model"] = override

    config["providers"] = providers
    config["roles"] = roles
    config.setdefault("quota_reservation", {})
    return config


@functools.lru_cache(maxsize=1)
def load_sector_config() -> dict[str, Any]:
    """Display strings and thresholds per sector, with environment overrides applied."""
    config = _read_yaml(CONFIG_DIR / "sectors.yaml")
    settings = get_settings()

    if settings.trade_approval_threshold_usd is not None:
        config["quant"]["thresholds"]["trade_approval_usd"] = settings.trade_approval_threshold_usd
    if settings.ecom_high_value_threshold_usd is not None:
        config["ecommerce"]["thresholds"]["high_value_usd"] = settings.ecom_high_value_threshold_usd

    for sector_id, entry in config.items():
        for required in ("display_name", "tagline", "accent", "disclaimer"):
            if not entry.get(required):
                raise ValueError(f"sector '{sector_id}' is missing '{required}'")
        entry.setdefault("thresholds", {})
    return config


def sector_settings(sector_id: str) -> dict[str, Any]:
    """Configuration for one sector, raising a clear error for an unknown id."""
    config = load_sector_config()
    if sector_id not in config:
        raise KeyError(f"unknown sector '{sector_id}'; known sectors: {sorted(config)}")
    return config[sector_id]


def sector_threshold(sector_id: str, name: str, default: Any = None) -> Any:
    return sector_settings(sector_id)["thresholds"].get(name, default)

In [ ]:
import datetime as dt
import json
import logging
import re
import sys

LOGGER_NAME = "automatron"

# Provider key shapes, checked first so a key is never mistaken for a card number.
_KEY_PATTERN = re.compile(
    r"\b(?:gsk_[A-Za-z0-9]{10,}"
    r"|sk-or-v1-[A-Za-z0-9]{10,}"
    r"|sk-[A-Za-z0-9]{20,}"
    r"|AIza[0-9A-Za-z_-]{10,}"
    # Google also issues keys in an "AQ.<base64ish>" form that shares no prefix
    # with the AIza style, so both shapes have to be listed.
    r"|AQ\.[A-Za-z0-9_-]{20,}"
    r"|csk-[A-Za-z0-9]{10,})"
)
_EMAIL_PATTERN = re.compile(
    r"\b([A-Za-z0-9._%+-])[A-Za-z0-9._%+-]*@([A-Za-z0-9-])[A-Za-z0-9.-]*\.([A-Za-z]{2,})\b"
)
# 13 to 19 digits, optionally grouped, which covers the common card formats.
_CARD_PATTERN = re.compile(r"(?<![\d-])(?:\d[ -]?){12,18}\d(?![\d-])")
_PHONE_PATTERN = re.compile(
    r"(?<![\d.])(?:\+\d{1,3}[ -]?)?(?:\(\d{3}\)|\d{3})[ -]\d{3}[ -]\d{4}(?![\d.])"
)


def _mask_email(match: re.Match[str]) -> str:
    return f"{match.group(1)}***@{match.group(2)}***.{match.group(3)}"


def _mask_card(match: re.Match[str]) -> str:
    digits = re.sub(r"\D", "", match.group(0))
    return f"****{digits[-4:]}"


def _mask_phone(match: re.Match[str]) -> str:
    digits = re.sub(r"\D", "", match.group(0))
    return f"***{digits[-2:]}"


def redact(text: str) -> str:
    """Mask credentials and personal data so they never reach a log or a trace.

    Applied to every log record and to any text shown in the interface, so it has
    to be cheap and has to leave ordinary numbers alone.
    """
    if not text:
        return text
    masked = _KEY_PATTERN.sub("[redacted-key]", text)
    masked = _EMAIL_PATTERN.sub(_mask_email, masked)
    masked = _CARD_PATTERN.sub(_mask_card, masked)
    return _PHONE_PATTERN.sub(_mask_phone, masked)


class JsonFormatter(logging.Formatter):
    """One JSON object per line, which is what both hosting platforms collect."""

    EXTRA_FIELDS = (
        "run_id",
        "role",
        "provider",
        "model",
        "step_id",
        "latency_ms",
        "outcome",
        "reason",
    )

    def format(self, record: logging.LogRecord) -> str:
        payload: dict[str, Any] = {
            "ts": dt.datetime.fromtimestamp(record.created, dt.UTC).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": redact(record.getMessage()),
        }
        for field in self.EXTRA_FIELDS:
            value = getattr(record, field, None)
            if value is not None:
                payload[field] = redact(value) if isinstance(value, str) else value
        if record.exc_info:
            payload["error"] = redact(self.formatException(record.exc_info))
        return json.dumps(payload, default=str)


def setup_logging(level: str | None = None) -> logging.Logger:
    """Send redacted JSON logs to stdout. Safe to call more than once."""
    logger = logging.getLogger(LOGGER_NAME)
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(JsonFormatter())
    logger.handlers = [handler]
    logger.setLevel((level or get_settings().log_level).upper())
    logger.propagate = False
    return logger


def get_logger(name: str | None = None) -> logging.Logger:
    """Return a child of the application logger, configuring it on first use."""
    root = logging.getLogger(LOGGER_NAME)
    if not root.handlers:
        setup_logging()
    return root.getChild(name) if name else root

## 2. Schemas

Plan, PlanStep, StepResult and Evidence for the handoff contract; DecisionBrief,
ApprovalDecision and TraceEvent for the gate and the trace; SectorPack and
WorkflowSpec for what each sector contributes.

In [ ]:
from collections.abc import Callable

from pydantic import BaseModel, ConfigDict, Field, field_validator

EvidenceKind = Literal["tool", "source", "upload"]
AgentName = Literal["researcher", "analyst", "executor"]
StepStatus = Literal["ok", "partial", "failed"]
Severity = Literal["info", "low", "medium", "high", "critical"]
Confidence = Literal["low", "medium", "high"]
DecisionAction = Literal["approve", "request_changes", "reject"]
TraceKind = Literal["start", "tool_call", "tool_result", "failover", "done", "warning", "error"]
RunStatus = Literal[
    "queued", "running", "awaiting_approval", "revising", "approved", "rejected", "failed"
]

MIN_PLAN_STEPS = 2
MAX_PLAN_STEPS = 7
MAX_EXCERPT_CHARS = 300
MAX_TRACE_MESSAGE_CHARS = 160

# The only brief fields a reviewer may edit by hand at the approval gate.
EDITABLE_BRIEF_FIELDS = ("recommendation", "options", "reviewer_comments")


def utcnow_iso() -> str:
    return dt.datetime.now(dt.UTC).isoformat(timespec="seconds")


class Evidence(BaseModel):
    """A pointer back to the tool output, retrieved source, or upload behind a claim."""

    id: str
    kind: EvidenceKind
    label: str
    locator: str | None = None
    excerpt: str | None = None

    @field_validator("excerpt")
    @classmethod
    def _cap_excerpt(cls, value: str | None) -> str | None:
        # Truncate rather than reject: evidence arrives from tools mid-run.
        return None if value is None else value[:MAX_EXCERPT_CHARS]


class StepResultDraft(BaseModel):
    """What a sub-agent writes. The tool loop fills in the runtime fields."""

    status: StepStatus = "ok"
    summary: str
    data: dict[str, Any] = Field(default_factory=dict)
    missing_inputs: list[str] = Field(default_factory=list)


class StepResult(BaseModel):
    """The handoff contract between a sub-agent and the coordinator."""

    step_id: str
    agent: AgentName
    status: StepStatus
    summary: str
    data: dict[str, Any] = Field(default_factory=dict)
    evidence: list[Evidence] = Field(default_factory=list)
    missing_inputs: list[str] = Field(default_factory=list)
    provider: str = ""
    model: str = ""
    tool_calls: int = 0
    latency_ms: int = 0


class PlanStep(BaseModel):
    id: str
    agent: AgentName
    instruction: str
    tool_hints: list[str] = Field(default_factory=list)
    depends_on: list[str] = Field(default_factory=list)
    expected_output: str = ""


class Plan(BaseModel):
    objective: str
    steps: list[PlanStep]
    notes: str | None = None

    def step_ids(self) -> list[str]:
        return [step.id for step in self.steps]

    def issues(self) -> list[str]:
        """Report everything wrong with this plan, so one repair call can fix it all."""
        problems: list[str] = []
        ids = self.step_ids()

        if not MIN_PLAN_STEPS <= len(self.steps) <= MAX_PLAN_STEPS:
            problems.append(
                f"a plan needs between {MIN_PLAN_STEPS} and {MAX_PLAN_STEPS} steps, "
                f"got {len(self.steps)}"
            )
        duplicates = sorted({step_id for step_id in ids if ids.count(step_id) > 1})
        if duplicates:
            problems.append(f"duplicate step ids: {duplicates}")

        known = set(ids)
        for step in self.steps:
            unknown = [dep for dep in step.depends_on if dep not in known]
            if unknown:
                problems.append(f"step '{step.id}' depends on unknown steps: {unknown}")
            if step.id in step.depends_on:
                problems.append(f"step '{step.id}' depends on itself")

        if not problems and self.has_cycle():
            problems.append("plan steps form a dependency cycle")
        return problems

    def has_cycle(self) -> bool:
        """Kahn's algorithm: whatever cannot be ordered is part of a cycle."""
        pending = {step.id: set(step.depends_on) for step in self.steps}
        while True:
            ready = [step_id for step_id, deps in pending.items() if not deps]
            if not ready:
                return bool(pending)
            for step_id in ready:
                pending.pop(step_id)
            for deps in pending.values():
                deps.difference_update(ready)


class Finding(BaseModel):
    text: str
    severity: Severity = "info"
    evidence_ids: list[str] = Field(default_factory=list)


class Option(BaseModel):
    name: str
    description: str
    pros: list[str] = Field(default_factory=list)
    cons: list[str] = Field(default_factory=list)


class DecisionBrief(BaseModel):
    """The deliverable: a proposal for a human reviewer, never a decision."""

    title: str
    sector: str
    workflow_id: str
    summary: str
    recommendation: str
    recommendation_level: str
    confidence: Confidence = "low"
    confidence_reason: str = ""
    key_findings: list[Finding] = Field(default_factory=list)
    quantitative_results: dict[str, str] = Field(default_factory=dict)
    data_quality_issues: list[str] = Field(default_factory=list)
    missing_information: list[str] = Field(default_factory=list)
    options: list[Option] = Field(default_factory=list)
    reviewer_must_decide: str = ""
    drafts: list[dict[str, Any]] = Field(default_factory=list)
    evidence: list[Evidence] = Field(default_factory=list)
    disclaimer: str = ""
    revision_notes: list[str] = Field(default_factory=list)
    verification_warnings: list[str] = Field(default_factory=list)
    reviewer_comments: str | None = None
    decided_by: str | None = None
    decided_at: str | None = None
    decision: Literal["approved", "rejected"] | None = None


class ApprovalDecision(BaseModel):
    """What the reviewer submits at the approval gate."""

    action: DecisionAction
    reviewer: str = Field(min_length=1, max_length=80)
    notes: str = ""
    edits: dict[str, Any] | None = None

    @field_validator("reviewer")
    @classmethod
    def _require_name(cls, value: str) -> str:
        cleaned = value.strip()
        if not cleaned:
            raise ValueError("a reviewer name is required")
        return cleaned

    @field_validator("edits")
    @classmethod
    def _only_editable_fields(cls, value: dict[str, Any] | None) -> dict[str, Any] | None:
        if value is None:
            return None
        rejected = sorted(set(value) - set(EDITABLE_BRIEF_FIELDS))
        if rejected:
            raise ValueError(f"these brief fields cannot be edited by hand: {rejected}")
        return value


class TraceEvent(BaseModel):
    """One line in the visible agent trace."""

    run_id: str
    node: str
    kind: TraceKind
    ts: str = Field(default_factory=utcnow_iso)
    agent: str | None = None
    provider: str | None = None
    model: str | None = None
    step_id: str | None = None
    message: str = ""
    latency_ms: int | None = None

    @field_validator("message")
    @classmethod
    def _short_and_safe(cls, value: str) -> str:
        return redact(value)[:MAX_TRACE_MESSAGE_CHARS]

In [ ]:
class ToolSpec(BaseModel):
    """A tool plus the roles allowed to call it. The allowlist is enforced in code."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    tool: Any
    roles: list[AgentName]

    @property
    def name(self) -> str:
        return getattr(self.tool, "name", None) or getattr(self.tool, "__name__", "")


class WorkflowSpec(BaseModel):
    """One of the twelve workflows: its inputs, its fallback plan, and its vocabulary."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    id: str
    name: str
    description: str
    input_schema: type[BaseModel]
    accepted_uploads: list[str] = Field(default_factory=list)
    step_template: str = ""
    default_plan: Plan
    fake_script: dict[str, Any] = Field(default_factory=dict)
    level_vocab: list[str] = Field(default_factory=list)
    forbidden_phrases: list[str] = Field(default_factory=list)
    sample_name: str = ""
    example_request: str = ""

    @field_validator("default_plan")
    @classmethod
    def _fallback_plan_must_be_usable(cls, value: Plan) -> Plan:
        # This plan runs whenever the coordinator's own plan fails validation, so a
        # broken one would only surface during an outage.
        problems = value.issues()
        if problems:
            raise ValueError(f"default_plan is not a valid plan: {problems}")
        return value


class SectorPack(BaseModel):
    """Everything one sector contributes: tools, workflows, and prompt guidance."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    id: str
    display_name: str
    tagline: str
    accent: str
    disclaimer: str
    tools: list[ToolSpec] = Field(default_factory=list)
    workflows: list[WorkflowSpec] = Field(default_factory=list)
    addenda: dict[str, str] = Field(default_factory=dict)
    ensure_samples: Callable[[], None] | None = None

    def workflow(self, workflow_id: str) -> WorkflowSpec:
        for spec in self.workflows:
            if spec.id == workflow_id:
                return spec
        known = [spec.id for spec in self.workflows]
        raise KeyError(f"sector '{self.id}' has no workflow '{workflow_id}'; known: {known}")

    def has_workflow(self, workflow_id: str) -> bool:
        return any(spec.id == workflow_id for spec in self.workflows)

    def tools_for(self, role: str) -> list[Any]:
        """The tools one role may call. Anything outside this list is refused."""
        return [spec.tool for spec in self.tools if role in spec.roles]

    def tool_names_for(self, role: str) -> set[str]:
        return {spec.name for spec in self.tools if role in spec.roles}

    def addendum(self, role: str) -> str:
        return self.addenda.get(role, "")


def sector_identity(sector_id: str) -> dict[str, str]:
    """Display fields for a sector pack, read from config rather than hardcoded."""
    entry = sector_settings(sector_id)
    return {
        "id": sector_id,
        "display_name": entry["display_name"],
        "tagline": entry["tagline"],
        "accent": entry["accent"],
        # The YAML folds long disclaimers across lines; collapse them for display.
        "disclaimer": " ".join(entry["disclaimer"].split()),
    }

## 3. Sector registry

register_sector, get_sector, list_sectors.

In [ ]:
_SECTOR_REGISTRY: dict[str, SectorPack] = {}


class UnknownSector(KeyError):
    """Raised when a request names a sector that no pack has registered."""


def register_sector(pack: SectorPack) -> SectorPack:
    """Add a sector pack to the registry. Importing a sector notebook calls this."""
    if pack.id not in SECTOR_ORDER:
        raise ValueError(f"'{pack.id}' is not a known sector; expected one of {SECTOR_ORDER}")

    workflow_ids = [spec.id for spec in pack.workflows]
    duplicates = sorted({wid for wid in workflow_ids if workflow_ids.count(wid) > 1})
    if duplicates:
        raise ValueError(f"sector '{pack.id}' registers duplicate workflow ids: {duplicates}")
    misnamed = [wid for wid in workflow_ids if not wid.startswith(f"{pack.id}.")]
    if misnamed:
        raise ValueError(f"workflow ids must be prefixed with '{pack.id}.': {misnamed}")

    if pack.id in _SECTOR_REGISTRY:
        get_logger("registry").info("replacing already registered sector '%s'", pack.id)
    _SECTOR_REGISTRY[pack.id] = pack
    return pack


def get_sector(sector_id: str) -> SectorPack:
    try:
        return _SECTOR_REGISTRY[sector_id]
    except KeyError:
        raise UnknownSector(
            f"sector '{sector_id}' is not registered; registered: {sorted(_SECTOR_REGISTRY)}"
        ) from None


def list_sectors() -> list[SectorPack]:
    """Registered packs in display order, so the interface never has to sort them."""
    return [_SECTOR_REGISTRY[key] for key in SECTOR_ORDER if key in _SECTOR_REGISTRY]


def registered_sector_ids() -> list[str]:
    return [pack.id for pack in list_sectors()]


def get_workflow(sector_id: str, workflow_id: str) -> WorkflowSpec:
    return get_sector(sector_id).workflow(workflow_id)


def clear_registry() -> None:
    """Empty the registry. Tests use this to install a pack of their own."""
    _SECTOR_REGISTRY.clear()

## 4. Provider router

ProviderRouter, ProviderSlot, error classification, cooldowns, FakeChatModel.

## 5. Retrieval layer

Embeddings, Qdrant client, ingestion, search_knowledge, session cleanup.

## 6. Tool-agent loop

run_tool_agent: provider-agnostic tool calling with a structured finish.

## 7. Graph builder

build_graph(sector_pack): the supervisor state machine.

## 8. Verifier, rendering and audit

Brief verification, Markdown rendering, hash-chained audit log.

## 9. Run service

start_run, get_run, submit_decision, stream_events.

## 10. HTTP API

FastAPI app factory and REST routes.

## 11. User interface

Gradio interface and create_app().

In [ ]:
PLACEHOLDER_PAGE = """<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Automatron</title>
<style>
  :root {
    --bg: #0B0D12; --surface: #12151C; --border: #232836;
    --text: #E6E8EE; --muted: #8A93A6; --accent: #7C8CFF;
  }
  * { box-sizing: border-box; }
  body {
    margin: 0; min-height: 100vh; display: grid; place-items: center; padding: 24px;
    background: var(--bg); color: var(--text);
    font-family: Inter, system-ui, sans-serif;
  }
  main {
    max-width: 520px; padding: 32px;
    background: var(--surface); border: 1px solid var(--border); border-radius: 14px;
  }
  h1 { margin: 0 0 8px; font-size: 1.5rem; font-weight: 600; letter-spacing: -0.01em; }
  h1 span { color: var(--accent); }
  p { margin: 0; color: var(--muted); line-height: 1.6; }
  .tag {
    display: inline-block; margin-bottom: 20px; padding: 4px 12px;
    border: 1px solid var(--border); border-radius: 999px;
    color: var(--muted); font-size: 0.75rem;
  }
</style>
</head>
<body>
  <main>
    <span class="tag">starting up</span>
    <h1>auto<span>matron</span></h1>
    <p>Multi-agent decision support for space, quant, e-commerce and real estate.
       The interface is not mounted yet.</p>
  </main>
</body>
</html>
"""


def create_app():
    """Build the FastAPI application that hosts the API and the interface.

    Imports stay inside the function so that importing this module stays cheap
    and free of side effects.
    """
    from fastapi import FastAPI
    from fastapi.responses import HTMLResponse

    app = FastAPI(title="Automatron", docs_url=None, redoc_url=None)

    @app.get("/", response_class=HTMLResponse)
    def index() -> str:
        return PLACEHOLDER_PAGE

    return app

## Exports

The public surface a sector notebook receives from a star import.

In [ ]:
# Sector notebooks do `from automatron_core import *`, so this list is the core
# module's public surface. Pydantic names are re-exported so a sector notebook can
# declare its input schemas without importing pydantic itself.
__all__ = [
    "AGENT_ROLES",
    "AgentName",
    "ApprovalDecision",
    "BaseModel",
    "CONFIG_DIR",
    "ConfigDict",
    "Confidence",
    "DecisionAction",
    "DecisionBrief",
    "EDITABLE_BRIEF_FIELDS",
    "Evidence",
    "EvidenceKind",
    "Field",
    "Finding",
    "JsonFormatter",
    "LOGGER_NAME",
    "MAX_PLAN_STEPS",
    "MIN_PLAN_STEPS",
    "Option",
    "PLACEHOLDER_PAGE",
    "PROVIDER_ORDER",
    "Plan",
    "PlanStep",
    "ROOT",
    "RunStatus",
    "SECTOR_ORDER",
    "SectorPack",
    "Settings",
    "Severity",
    "StepResult",
    "StepResultDraft",
    "StepStatus",
    "ToolSpec",
    "TraceEvent",
    "TraceKind",
    "UnknownSector",
    "WorkflowSpec",
    "clear_registry",
    "create_app",
    "field_validator",
    "get_logger",
    "get_sector",
    "get_settings",
    "get_workflow",
    "hello",
    "list_sectors",
    "load_provider_config",
    "load_sector_config",
    "redact",
    "register_sector",
    "registered_sector_ids",
    "reset_settings_cache",
    "sector_identity",
    "sector_settings",
    "sector_threshold",
    "setup_logging",
    "utcnow_iso",
]

## Build check

Confirms the notebook reached the generated module cleanly.

In [ ]:
def hello() -> str:
    """Return this module's name, so the build pipeline can be checked end to end."""
    return "automatron_core"